# MASW-YOLO: آموزش مدل پایه YOLOv8n روی VisDrone2019

**مطابق تنظیمات جدول 1 مقاله MASW-YOLO**

| پارامتر | مقدار | 
|---------|-------|
| epochs  | 100   |
| batch   | 16    |
| imgsz   | 640   |
| lr0     | 0.01  |
| lrf     | 0.01  |
| momentum| 0.937 |

## 1. بررسی GPU
مطابق مقاله از NVIDIA GeForce GTX 1660 SUPER استفاده شده. در Colab از Tesla T4 یا V100 استفاده می‌شود.

In [ ]:
!nvidia-smi

## 2. نصب کتابخانه Ultralytics
مطابق مقاله: Python 3.8, torch 1.12.0, CUDA 11.3

In [ ]:
!pip install -U ultralytics

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Ultralytics version: {ultralytics.__version__}')

## 3. اتصال به Google Drive
دیتاست VisDrone2019 باید در گوگل درایو شما قرار داشته باشد.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. بررسی ساختار دیتاست
بررسی وجود دیتاست VisDrone2019 در Google Drive

In [ ]:
import os

# مسیر دیتاست در Google Drive - در صورت نیاز ویرایش کنید
DATASET_PATH = '/content/drive/MyDrive/VisDrone2019'

# بررسی وجود پوشه‌ها
if os.path.exists(DATASET_PATH):
    print(f'✅ دیتاست در مسیر زیر یافت شد: {DATASET_PATH}')
    for split in ['train', 'val', 'test']:
        img_dir = os.path.join(DATASET_PATH, 'images', split)
        lbl_dir = os.path.join(DATASET_PATH, 'labels', split)
        if os.path.exists(img_dir):
            n_imgs = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
            print(f'   {split}: {n_imgs} تصویر')
        else:
            print(f'   ⚠️ پوشه {split} تصاویر یافت نشد: {img_dir}')
        if os.path.exists(lbl_dir):
            n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
            print(f'   {split}: {n_lbls} برچسب')
        else:
            print(f'   ⚠️ پوشه {split} برچسب‌ها یافت نشد: {lbl_dir}')
else:
    print(f'❌ دیتاست یافت نشد: {DATASET_PATH}')
    print('لطفاً دیتاست VisDrone2019 را در گوگل درایو خود آپلود کنید.')
    print('ساختار مورد انتظار:')
    print('  /content/drive/MyDrive/VisDrone2019/')
    print('  ├── images/')
    print('  │   ├── train/')
    print('  │   ├── val/')
    print('  │   └── test/')
    print('  └── labels/')
    print('      ├── train/')
    print('      ├── val/')
    print('      └── test/')

## 5. ایجاد فایل کانفیگ دیتاست
فایل YAML برای معرفی دیتاست به YOLOv8

In [ ]:
import yaml

# ایجاد فایل کانفیگ دیتاست
dataset_config = {
    'path': '/content/drive/MyDrive/VisDrone2019',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'pedestrian',
        1: 'people',
        2: 'bicycle',
        3: 'car',
        4: 'van',
        5: 'truck',
        6: 'tricycle',
        7: 'awning-tricycle',
        8: 'bus',
        9: 'motor'
    }
}

yaml_path = '/content/visdrone.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, allow_unicode=True)

print(f'✅ فایل کانفیگ دیتاست ایجاد شد: {yaml_path}')
with open(yaml_path, 'r') as f:
    print(f.read())

## 6. بارگذاری مدل پایه
YOLOv8n با وزن‌های پیش‌آموزش‌دیده روی COCO

In [ ]:
# بارگذاری مدل پایه YOLOv8n
model = YOLO('yolov8n.pt')

print('✅ مدل YOLOv8n بارگذاری شد')
print(f'تعداد پارامترها: {sum(p.numel() for p in model.model.parameters()) / 1e6:.2f}M')

## 7. آموزش مدل (مطابق جدول 1 مقاله)

تنظیمات دقیق:
- **epochs=100** (جدول 1 مقاله)
- **batch=16** (جدول 1 مقاله)
- **imgsz=640** (جدول 1 مقاله)
- **lr0=0.01** (جدول 1 مقاله)
- **lrf=0.01** (جدول 1 مقاله)
- **momentum=0.937** (جدول 1 مقاله)
- **weight_decay=0.0005** (پیش‌فرض YOLOv8)

In [ ]:
# شروع آموزش با تنظیمات دقیق مقاله
results = model.train(
    data='/content/visdrone.yaml',      # فایل تنظیمات دیتاست
    epochs=100,                          # مطابق جدول 1 مقاله
    batch=16,                            # مطابق جدول 1 مقاله
    imgsz=640,                           # مطابق جدول 1 مقاله
    lr0=0.01,                            # مطابق جدول 1 مقاله
    lrf=0.01,                            # مطابق جدول 1 مقاله
    momentum=0.937,                      # مطابق جدول 1 مقاله
    weight_decay=0.0005,                 # پیش‌فرض YOLOv8
    warmup_epochs=3.0,                   # پیش‌فرض YOLOv8
    warmup_momentum=0.8,                 # پیش‌فرض YOLOv8
    warmup_bias_lr=0.1,                  # پیش‌فرض YOLOv8
    optimizer='SGD',                     # بهینه‌ساز
    device=0,                            # GPU
    workers=8,                           # تعداد کارگران
    project='runs/train',                # مسیر ذخیره نتایج
    name='yolov8n_visdrone_baseline',    # نام آزمایش
    exist_ok=True,                       # بازنویسی در صورت تکرار
    verbose=True                         # نمایش جزئیات
)

print('\n✅ آموزش با موفقیت به پایان رسید!')

## 8. ارزیابی مدل
محاسبه mAP@0.5 و mAP@0.5-0.95

In [ ]:
# ارزیابی روی مجموعه اعتبارسنجی
metrics = model.val()

print('\n📊 نتایج ارزیابی مدل پایه YOLOv8n:')
print(f'   Precision:  {metrics.box.mp:.4f}')
print(f'   Recall:     {metrics.box.mr:.4f}')
print(f'   mAP@0.5:    {metrics.box.map50:.4f}  (انتظار مقاله: 0.304)')
print(f'   mAP@0.5-95: {metrics.box.map:.4f}  (انتظار مقاله: 0.174)')

## 9. ذخیره مدل در Google Drive
ذخیره وزن‌های آموزش‌دیده برای استفاده‌های بعدی

In [ ]:
import shutil

# مسیر ذخیره در Google Drive
save_dir = '/content/drive/MyDrive/MASW-YOLO-Results/baseline'
os.makedirs(save_dir, exist_ok=True)

# کپی بهترین وزن‌ها
best_weights = 'runs/train/yolov8n_visdrone_baseline/weights/best.pt'
if os.path.exists(best_weights):
    shutil.copy(best_weights, os.path.join(save_dir, 'best.pt'))
    print(f'✅ بهترین وزن‌ها در Google Drive ذخیره شد: {save_dir}/best.pt')

# کپی وزن‌های آخر epoch
last_weights = 'runs/train/yolov8n_visdrone_baseline/weights/last.pt'
if os.path.exists(last_weights):
    shutil.copy(last_weights, os.path.join(save_dir, 'last.pt'))
    print(f'✅ آخرین وزن‌ها در Google Drive ذخیره شد: {save_dir}/last.pt')

# کپی نتایج
results_dir = 'runs/train/yolov8n_visdrone_baseline'
if os.path.exists(results_dir):
    for item in os.listdir(results_dir):
        src = os.path.join(results_dir, item)
        if os.path.isfile(src) and item.endswith(('.png', '.csv', '.yaml')):
            shutil.copy(src, save_dir)
    print(f'✅ نتایج در Google Drive ذخیره شد: {save_dir}')

## 10. مقایسه با نتایج مقاله

| مدل | Params(M) | FLOPs(G) | P(%) | Recall(%) | mAP@0.5(%) | mAP@0.5-95(%) |
|-----|-----------|----------|------|-----------|------------|---------------|
| YOLOv8n (مقاله) | 3.01 | 8.1 | 41.4 | 31.0 | 30.4 | 17.4 |
| YOLOv8n (شما) | - | - | - | - | - | - |

In [ ]:
# نمایش نتایج نهایی برای مقایسه
print('='*60)
print('📊 مقایسه نتایج شما با مقاله:')
print('='*60)
print(f'{"متریک":<20} {"مقاله":<15} {"شما":<15}')
print('-'*60)
print(f'{"mAP@0.5 (%)":<20} {"30.4":<15} {metrics.box.map50*100:<15.1f}')
print(f'{"mAP@0.5-95 (%)":<20} {"17.4":<15} {metrics.box.map*100:<15.1f}')
print(f'{"Precision (%)":<20} {"41.4":<15} {metrics.box.mp*100:<15.1f}')
print(f'{"Recall (%)":<20} {"31.0":<15} {metrics.box.mr*100:<15.1f}')
print('='*60)